# S2: Label Efficiency Sweep

**Leaf-JEPA IRP** | Stage 5 — PEFT Adaptation Experiments

**Research Questions:**
- RQ3 — Can best PEFT match full FT (within 2%) at <2% params across 1–100% labels?
- RQ5 — Does Leaf-JEPA + PEFT outperform supervised CNNs at ≤10% labels?

**Prerequisite:** Run S1 first. This notebook selects the top 2 PEFT methods from S1 results.

**Key Output:** The label efficiency curve — the single most important figure in the dissertation.

## Imports & Configuration

In [ ]:
import sys
import json
from pathlib import Path
from collections import defaultdict

PROJECT_ROOT = Path("..").resolve().parent
sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np

from stage5_peft_adaptation_experiments.config_stage5 import *
from stage5_peft_adaptation_experiments.peft_utils import (
    set_seed, get_device, train_peft, aggregate_seed_results,
    save_results, load_results, plot_label_efficiency
)

verify_config()
device = get_device()

SET2_DIR = PEFT_DIR / "set2"
SET2_DIR.mkdir(parents=True, exist_ok=True)

## Select Top Methods from Set 1

In [2]:
# Load Set 1 results
set1_results = load_results(PEFT_DIR / "set1_method_comparison.json")

# Aggregate by (method, hp_tag)
method_scores = defaultdict(list)
for res in set1_results:
    key = (res["method"], res["hp_tag"])
    method_scores[key].append(res["results"]["test_macro_f1"])

# Rank by mean F1
print("Set 1 Results by Method (mean ± std across seeds):")
print("-" * 60)

ranked = []
for (method, tag), scores in method_scores.items():
    mean_f1 = np.mean(scores)
    std_f1 = np.std(scores)
    ranked.append((method, tag, mean_f1, std_f1))
    print(f"  {method:<15} ({tag:<5}) | F1: {mean_f1:.4f} ± {std_f1:.4f}")

ranked.sort(key=lambda x: x[2], reverse=True)

# Select top 2
TOP_METHODS = [
    {"method": ranked[0][0], "hp_tag": ranked[0][1]},
    {"method": ranked[1][0], "hp_tag": ranked[1][1]},
]

print(f"\n★ Selected for label efficiency sweep:")
for i, m in enumerate(TOP_METHODS, 1):
    print(f"  {i}. {m['method']} ({m['hp_tag']})")

FileNotFoundError: [Errno 2] No such file or directory: 'D:\\IRP\\leaf-jepa\\stage5_peft_adaptation_experiments\\outputs\\peft\\set1_method_comparison.json'

## Build Experiment Grid

In [ ]:
def hp_tag_to_kwargs(method, hp_tag):
    """Convert hp_tag string back to method kwargs."""
    if method == "lora":
        return {"rank": int(hp_tag.replace("r", ""))}
    elif method == "adapter":
        return {"bottleneck_dim": int(hp_tag.replace("d", ""))}
    elif method in ("vpt_shallow", "vpt_deep"):
        kwargs = {"num_prompts": int(hp_tag.replace("p", ""))}
        if method == "vpt_deep":
            kwargs["num_layers"] = VIT_DEPTH
        return kwargs
    elif method == "bitfit":
        return {}
    return {}


# Build experiment list
experiments = []
for m_info in TOP_METHODS:
    method = m_info["method"]
    hp_tag = m_info["hp_tag"]
    kwargs = hp_tag_to_kwargs(method, hp_tag)
    
    for frac in LABEL_FRACTIONS:
        for seed in SUBSET_SEEDS:
            experiments.append({
                "method": method,
                "hp_tag": hp_tag,
                "kwargs": kwargs,
                "fraction": frac,
                "seed": seed,
            })

print(f"Total label efficiency runs: {len(experiments)}")
print(f"  {len(TOP_METHODS)} methods × {len(LABEL_FRACTIONS)} fractions × {len(SUBSET_SEEDS)} seeds")

## Run Label Efficiency Sweep

In [ ]:
all_results = []

for i, exp in enumerate(experiments):
    run_name = wandb_run_name(
        exp["method"], exp["hp_tag"], exp["fraction"], exp["seed"]
    )
    
    print(f"\n[{i+1}/{len(experiments)}] {run_name}")
    
    result = train_peft(
        method=exp["method"],
        checkpoint_path=LEAF_JEPA_CHECKPOINT,
        pv_root=PV_ROOT,
        splits_dir=SPLITS_DIR,
        norm_mean=NORM_MEAN,
        norm_std=NORM_STD,
        model_name=VIT_MODEL_NAME,
        embed_dim=VIT_EMBED_DIM,
        num_classes=NUM_CLASSES,
        fraction=exp["fraction"],
        seed=exp["seed"],
        lr=BEST_LR[exp["method"]],
        batch_size=BATCH_SIZE,
        max_epochs=MAX_EPOCHS,
        patience=EARLY_STOP_PATIENCE,
        weight_decay=WEIGHT_DECAY,
        warmup_fraction=WARMUP_FRACTION,
        use_amp=USE_AMP,
        gradient_clip=GRADIENT_CLIP,
        num_workers=NUM_WORKERS,
        class_weights_path=CLASS_WEIGHTS_PATH,
        save_dir=SET2_DIR,
        run_name=run_name,
        wandb_project=WANDB_PROJECT,
        wandb_entity=WANDB_ENTITY,
        wandb_group=WANDB_GROUPS["set2"],
        **exp["kwargs"],
    )
    
    result["hp_tag"] = exp["hp_tag"]
    all_results.append(result)

save_results(all_results, PEFT_DIR / "set2_label_efficiency.json")
print(f"\n✅ Label efficiency sweep complete. {len(all_results)} results saved.")

## Aggregate Results & Plot Curves

In [ ]:
# Group results by (method, hp_tag, fraction)
grouped = defaultdict(list)
for res in all_results:
    key = (res["method"], res.get("hp_tag", ""), res["training_config"]["fraction"])
    grouped[key].append(res["results"]["test_macro_f1"])

# Structure for plotting: {label: {fraction: {mean, std}}}
plot_data = defaultdict(dict)
for (method, tag, frac), scores in grouped.items():
    label = f"Leaf-JEPA + {method} ({tag})"
    plot_data[label][frac] = {
        "mean": float(np.mean(scores)),
        "std": float(np.std(scores)),
    }

# Load Stage 3 baselines for comparison
baseline_curves_path = BASELINE_DIR / "label_efficiency_curves.json"
if baseline_curves_path.exists():
    baseline_curves = load_results(baseline_curves_path)
    for baseline_name, frac_data in baseline_curves.items():
        for frac_str, vals in frac_data.items():
            frac = float(frac_str)
            if baseline_name not in plot_data:
                plot_data[baseline_name] = {}
            plot_data[baseline_name][frac] = vals
    print("✅ Loaded Stage 3 baseline curves for comparison")
else:
    print("⚠️  Stage 3 label efficiency curves not found.")
    print(f"   Expected at: {baseline_curves_path}")
    print("   Add B1/B2 curves manually for the comparison figure.")

# Plot
plot_label_efficiency(
    plot_data,
    save_path=FIGURES_DIR / "set2_label_efficiency_curves.png",
    title="Label Efficiency: Leaf-JEPA + PEFT vs Baselines",
)
print(f"Saved: {FIGURES_DIR / 'set2_label_efficiency_curves.png'}")

## Crossover Analysis

In [ ]:
print("\nLabel Efficiency Results (mean ± std across 3 seeds)")
print("=" * 70)

for label, frac_data in sorted(plot_data.items()):
    print(f"\n  {label}:")
    for frac in sorted(frac_data.keys()):
        vals = frac_data[frac]
        m = vals["mean"]
        s = vals.get("std", 0)
        print(f"    {frac:6.0%} labels → F1 = {m:.4f} ± {s:.4f}")

# Identify crossover points
print("\n" + "=" * 70)
print("  CROSSOVER ANALYSIS")
print("=" * 70)
print("  (Identify where PEFT methods exceed baseline performance)")
print("  Compare the curves above to identify:")
print("  1. At what fraction does best PEFT exceed ResNet-50 at 100%?")
print("  2. At what fraction does best PEFT come within 2% of full FT (B4)?")

## Statistical Significance Tests

In [ ]:
from scipy import stats

print("\nStatistical Significance Tests")
print("=" * 60)
print("  Note: With 3 seeds, statistical power is limited.")
print("  Report Cohen's d effect size alongside p-values.")
print()

# Compare PEFT methods against each other at each fraction
for frac in LABEL_FRACTIONS:
    keys_at_frac = [
        k for k in grouped.keys() if k[2] == frac
    ]
    
    if len(keys_at_frac) >= 2:
        k1, k2 = keys_at_frac[0], keys_at_frac[1]
        scores1 = grouped[k1]
        scores2 = grouped[k2]
        
        label1 = f"{k1[0]} ({k1[1]})"
        label2 = f"{k2[0]} ({k2[1]})"
        
        # Effect size (Cohen's d)
        pooled_std = np.sqrt((np.std(scores1)**2 + np.std(scores2)**2) / 2)
        cohens_d = (np.mean(scores1) - np.mean(scores2)) / pooled_std if pooled_std > 0 else 0
        
        print(f"  Fraction {frac:.0%}:")
        print(f"    {label1}: {np.mean(scores1):.4f} ± {np.std(scores1):.4f}")
        print(f"    {label2}: {np.mean(scores2):.4f} ± {np.std(scores2):.4f}")
        print(f"    Cohen's d: {cohens_d:.3f}")
        print()

print("⚠️  Uncomment the t-test code below to compare against B1@100% from Stage 3.")
print("    Fill in b1_100_scores with actual values from Stage 3 results.")

# Example:
# b1_100_scores = [0.917, 0.919, 0.915]  # From Stage 3 B1 at 100%
# for (method, tag, frac), peft_scores in grouped.items():
#     if frac == 0.10:
#         t_stat, p_val = stats.ttest_ind(peft_scores, b1_100_scores)
#         print(f"  {method}@10% vs B1@100%: t={t_stat:.3f}, p={p_val:.4f}")